In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer import SwinTransformer3D
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# ============================================================
# Multi-seed 设置
# ============================================================
seeds = [42, 123, 777, 1004, 2024]
# [11, 22, 33, 44, 55]
patience = 30

all_seed_best_1mae = []
all_seed_best_epoch = []


# ============================================================
# 你最终只保留这一个文件
# ============================================================
best_model_path = os.path.join(
    "./save_swintransformer_folder",
    "best_model.pth"
)


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-08-24 17:07:45,715]::3827333478::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

In [5]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [6]:
train_set_data=getdata("/home/ssrlab/qx/intermediate_representation/data/full/train_set.dat","train")
print(1)
validation_set_data=getdata("/home/ssrlab/qx/intermediate_representation/data/full/valid_set.dat","valid")
print(1)

1
1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
train_dataloader = DataLoader(dataset=train_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
val_dataloader = DataLoader(dataset=val_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
max_value=0
print(1)

1


In [11]:
import os
import random
import numpy as np
import torch
from tqdm import tqdm


# ============================================================
# 固定随机种子
# ============================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 保证实验尽可能可复现
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# OCEAN 分离
# ============================================================
def extract_ocean(data, batch_size):
    o = []
    c = []
    e = []
    a = []
    n = []

    if batch_size > 1:

        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())

    elif batch_size == 1:

        data = data.squeeze()

        o.append(data[0].item())
        c.append(data[1].item())
        e.append(data[2].item())
        a.append(data[3].item())
        n.append(data[4].item())

    o = torch.tensor(o)
    c = torch.tensor(c)
    e = torch.tensor(e)
    a = torch.tensor(a)
    n = torch.tensor(n)

    return o, c, e, a, n




# ============================================================
# Multi-seed training
# ============================================================
for seed in seeds:

    print("\n" + "=" * 80)
    print(f"Start Seed: {seed}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. 设置当前 seed
    # --------------------------------------------------------
    set_seed(seed)


    # --------------------------------------------------------
    # 2. 每个 seed 都重新初始化模型
    # --------------------------------------------------------
    model = SwinTransformer3D()
    model.to(device)

    criterion = torch.nn.L1Loss().to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr
    )


    # --------------------------------------------------------
    # 如果是正常 multi-seed 实验，
    # 建议 test_flage=False
    # 否则会加载之前模型，破坏 seed 独立性
    # --------------------------------------------------------
    if test_flage == True:

        checkpoint = torch.load(
            best_model_path,
            map_location=device
        )

        model.load_state_dict(
            checkpoint["model"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer"]
        )


    # --------------------------------------------------------
    # 每个 seed 都必须重新初始化
    # --------------------------------------------------------
    max_value = float('inf')

    no_improve_count = 0

    best_epoch = 0


    with torch.cuda.device(0):

        for i in range(start_epoch, epochs):

            # =================================================
            # Train
            # =================================================
            model.train()

            train_avg_loss = 0

            train_avg_o_loss = 0
            train_avg_c_loss = 0
            train_avg_e_loss = 0
            train_avg_a_loss = 0
            train_avg_n_loss = 0


            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} Train Epoch {i + 1}"
            ):

                fullshot = fullshot.permute(
                    0, 4, 1, 2, 3
                )

                big_five_data = big_five_data.permute(
                    0, 2, 1
                )

                big_five_data = big_five_data.squeeze()


                audiodata = audiodata.to(device)

                fullshot = fullshot.to(device)

                big_five_data = big_five_data.to(device)


                optimizer.zero_grad()


                hypothesis = model(fullshot, audiodata)


                loss = criterion(
                    hypothesis,
                    big_five_data
                )


                loss.backward()

                optimizer.step()


                train_avg_loss += loss.item()


            train_avg_loss = (
                train_avg_loss /
                len(train_dataloader)
            )


            # =================================================
            # Validation
            # =================================================
            # model.eval()


            val_avg_loss = 0

            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0


            with torch.no_grad():

                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} Val Epoch {i + 1}"
                ):

                    fullshot = fullshot.permute(
                        0, 4, 1, 2, 3
                    )


                    big_five_data = big_five_data.permute(
                        0, 2, 1
                    )

                    big_five_data = big_five_data.squeeze()


                    audiodata = audiodata.to(device)

                    fullshot = fullshot.to(device)

                    big_five_data = big_five_data.to(device)


                    hypothesis = model(fullshot, audiodata)


                    val_loss = criterion(
                        hypothesis,
                        big_five_data
                    )


                    hypothesiso, \
                    hypothesisc, \
                    hypothesise, \
                    hypothesisa, \
                    hypothesisn = extract_ocean(
                        hypothesis,
                        len(fullshot)
                    )


                    big_five_datao, \
                    big_five_datac, \
                    big_five_datae, \
                    big_five_dataa, \
                    big_five_datan = extract_ocean(
                        big_five_data,
                        len(fullshot)
                    )


                    val_o_loss = criterion(
                        hypothesiso,
                        big_five_datao
                    )

                    val_c_loss = criterion(
                        hypothesisc,
                        big_five_datac
                    )

                    val_e_loss = criterion(
                        hypothesise,
                        big_five_datae
                    )

                    val_a_loss = criterion(
                        hypothesisa,
                        big_five_dataa
                    )

                    val_n_loss = criterion(
                        hypothesisn,
                        big_five_datan
                    )


                    val_avg_loss += val_loss.item()

                    val_avg_o_loss += val_o_loss.item()

                    val_avg_c_loss += val_c_loss.item()

                    val_avg_e_loss += val_e_loss.item()

                    val_avg_a_loss += val_a_loss.item()

                    val_avg_n_loss += val_n_loss.item()


            val_avg_loss = (
                val_avg_loss /
                len(val_dataloader)
            )

            val_avg_o_loss = (
                val_avg_o_loss /
                len(val_dataloader)
            )

            val_avg_c_loss = (
                val_avg_c_loss /
                len(val_dataloader)
            )

            val_avg_e_loss = (
                val_avg_e_loss /
                len(val_dataloader)
            )

            val_avg_a_loss = (
                val_avg_a_loss /
                len(val_dataloader)
            )

            val_avg_n_loss = (
                val_avg_n_loss /
                len(val_dataloader)
            )


            # =================================================
            # Best model
            #
            # 注意：
            # 永远保存到同一个 best_model.pth
            # 所以前一次会被覆盖
            # =================================================
            if val_avg_loss < max_value:

                max_value = val_avg_loss

                no_improve_count = 0

                best_epoch = i + 1


                torch.save(
                    {
                        'seed': seed,

                        'epoch': i + 1,

                        'model': model.state_dict(),

                        'optimizer': optimizer.state_dict(),

                        'loss': val_avg_loss,

                        'best_1_mae': 1 - val_avg_loss
                    },

                    best_model_path
                )


                logger.info(
                    '[Seed {}] New Best Model Saved | '
                    'Epoch: {} | '
                    '1-MAE: {:.4f}'.format(
                        seed,
                        i + 1,
                        1 - val_avg_loss
                    )
                )


            else:

                no_improve_count += 1


            # =================================================
            # 当前指标
            # =================================================
            current_1_mae = (
                1 - val_avg_loss
            )


            best_1_mae = (
                1 - max_value
            )


            logger.debug(
                'Seed: {} , '
                'Epoch: {} , '
                '1-MAE: {:.4f} , '
                'best 1-MAE: {:.4f} , '
                'patience: {}/{} , '
                '1-Training Loss: {:.4f} , '
                '1-val_avg_o_loss: {:.4f} , '
                '1-val_avg_c_loss: {:.4f} , '
                '1-val_avg_e_loss: {:.4f} , '
                '1-val_avg_a_loss: {:.4f} , '
                '1-val_avg_n_loss: {:.4f}'.format(

                    seed,

                    i + 1,

                    current_1_mae,

                    best_1_mae,

                    no_improve_count,

                    patience,

                    1 - train_avg_loss,

                    1 - val_avg_o_loss,

                    1 - val_avg_c_loss,

                    1 - val_avg_e_loss,

                    1 - val_avg_a_loss,

                    1 - val_avg_n_loss
                )
            )


            # =================================================
            # Early stopping
            # =================================================
            if no_improve_count >= patience:

                logger.info(
                    '[Seed {}] Early stopping triggered '
                    'at epoch {}. '
                    'Best epoch: {}. '
                    'Best 1-MAE: {:.4f}, '
                    'patience={}.'.format(

                        seed,

                        i + 1,

                        best_epoch,

                        best_1_mae,

                        patience
                    )
                )

                break


            torch.cuda.empty_cache()


    # ========================================================
    # 当前 seed 训练结束
    # ========================================================
    all_seed_best_1mae.append(
        best_1_mae
    )

    all_seed_best_epoch.append(
        best_epoch
    )


    print(
        f"\nSeed {seed} finished | "
        f"Best Epoch = {best_epoch} | "
        f"Best 1-MAE = {best_1_mae:.4f}"
    )


    # ========================================================
    # 删除模型，准备下一个 seed
    # ========================================================
    del model
    del optimizer

    torch.cuda.empty_cache()


# ============================================================
# 所有 seed 训练完成
# ============================================================
all_seed_best_1mae = np.array(
    all_seed_best_1mae
)


mean_1mae = np.mean(
    all_seed_best_1mae
)


std_1mae = np.std(
    all_seed_best_1mae,
    ddof=1
)


print("\n" + "=" * 80)
print("FINAL MULTI-SEED RESULTS")
print("=" * 80)


for seed, epoch, score in zip(
    seeds,
    all_seed_best_epoch,
    all_seed_best_1mae
):

    print(
        f"Seed {seed:>3} | "
        f"Best Epoch: {epoch:>3} | "
        f"Best 1-MAE: {score:.4f}"
    )


print("-" * 80)

print(
    f"Mean 1-MAE: {mean_1mae:.4f}"
)

print(
    f"Std  1-MAE: {std_1mae:.4f}"
)

print(
    f"Final Result: "
    f"{mean_1mae:.4f} ± {std_1mae:.4f}"
)

print("=" * 80)

print(
    f"\nFinal saved model: "
    f"{best_model_path}"
)


Start Seed: 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True


/home/ssrlab/ENTER/envs/qx/lib/python3.10/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


frequncey stride=10, time stride=10
number of patches=131


Seed 42 Train Epoch 1: 100%|█████████▉| 1497/1498 [06:16<00:00,  4.02it/s]/home/ssrlab/ENTER/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
Seed 42 Val Epoch 1: 100%|██████████| 498/498 [00:38<00:00, 12.79it/s]
[2026-08-24 17:14:50,884]::1670605970::INFO::[Seed 42] New Best Model Saved | Epoch: 1 | 1-MAE: 0.8804
[2026-08-24 17:14:50,884]::1670605970::DEBUG::Seed: 42 , Epoch: 1 , 1-MAE: 0.8804 , best 1-MAE: 0.8804 , patience: 0/30 , 1-Training Loss: 0.8491 , 1-val_avg_o_loss: 0.8690 , 1-val_avg_c_loss: 0.8870 , 1-val_avg_e_loss: 0.8970 , 1-val_avg_a_loss: 0.8706 , 1-val_avg_n_loss: 0.8782
Seed 42 Val Epoch 2: 100%|██████████| 498/498 [00:38<00:00, 12.78it/s]
[2026-08-24 17:21:48,879]::1670605970::INF


Seed 42 finished | Best Epoch = 110 | Best 1-MAE = 0.8987

Start Seed: 123
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


...


Seed 777 finished | Best Epoch = 5 | Best 1-MAE = 0.8958

Start Seed: 1004
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 1004 Val Epoch 1: 100%|██████████| 498/498 [00:39<00:00, 12.64it/s]
[2026-08-25 20:54:32,567]::1670605970::INFO::[Seed 1004] New Best Model Saved | Epoch: 1 | 1-MAE: 0.8560
[2026-08-25 20:54:32,568]::1670605970::DEBUG::Seed: 1004 , Epoch: 1 , 1-MAE: 0.8560 , best 1-MAE: 0.8560 , patience: 0/30 , 1-Training Loss: 0.8469 , 1-val_avg_o_loss: 0.8443 , 1-val_avg_c_loss: 0.8888 , 1-val_avg_e_loss: 0.7744 , 1-val_avg_a_loss: 0.8777 , 1-val_avg_n_loss: 0.8947
Seed 1004 Val Epoch 2: 100%|██████████| 498/498 [00:39<00:00, 12.61it/s]
[2026-08-25 21:01:35,608]::1670605970::INFO::[Seed 1004] New Best Model Saved | Epoch: 2 | 1-MAE: 0.8882
[2026-08-25 21:01:35,609]::1670605970::DEBUG::Seed: 1004 , Epoch: 2 , 1-MAE: 0.8882 , best 1-MAE: 0.8882 , patience: 0/30 , 1-Training Loss: 0.8707 , 1-val_avg_o_loss: 0.8931 , 1-val_avg_c_loss: 0.8916 , 1-val_avg_e_loss: 0.9035 , 1-val_avg_a_loss: 0.8575 , 1-val_avg_n_loss: 0.8952
Seed 1004 Val Epoch 3: 100%|██████████| 498/498 [00:39<00:00, 12.65it/s]
[2026


Seed 1004 finished | Best Epoch = 68 | Best 1-MAE = 0.9117

Start Seed: 2024
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 2024 Val Epoch 1: 100%|██████████| 498/498 [00:39<00:00, 12.64it/s]
[2026-08-26 08:21:48,720]::1670605970::INFO::[Seed 2024] New Best Model Saved | Epoch: 1 | 1-MAE: 0.8543
[2026-08-26 08:21:48,721]::1670605970::DEBUG::Seed: 2024 , Epoch: 1 , 1-MAE: 0.8543 , best 1-MAE: 0.8543 , patience: 0/30 , 1-Training Loss: 0.8477 , 1-val_avg_o_loss: 0.8700 , 1-val_avg_c_loss: 0.8492 , 1-val_avg_e_loss: 0.8961 , 1-val_avg_a_loss: 0.7884 , 1-val_avg_n_loss: 0.8678
Seed 2024 Val Epoch 2: 100%|██████████| 498/498 [00:39<00:00, 12.64it/s]
[2026-08-26 08:28:52,161]::1670605970::INFO::[Seed 2024] New Best Model Saved | Epoch: 2 | 1-MAE: 0.8931
[2026-08-26 08:28:52,162]::1670605970::DEBUG::Seed: 2024 , Epoch: 2 , 1-MAE: 0.8931 , best 1-MAE: 0.8931 , patience: 0/30 , 1-Training Loss: 0.8726 , 1-val_avg_o_loss: 0.8973 , 1-val_avg_c_loss: 0.8754 , 1-val_avg_e_loss: 0.9051 , 1-val_avg_a_loss: 0.8869 , 1-val_avg_n_loss: 0.9009
Seed 2024 Val Epoch 3: 100%|██████████| 498/498 [00:39<00:00, 12.63it/s]
[2026


Seed 2024 finished | Best Epoch = 17 | Best 1-MAE = 0.9118

FINAL MULTI-SEED RESULTS
Seed  42 | Best Epoch: 110 | Best 1-MAE: 0.8987
Seed 123 | Best Epoch:  53 | Best 1-MAE: 0.9112
Seed 777 | Best Epoch:   5 | Best 1-MAE: 0.8958
Seed 1004 | Best Epoch:  68 | Best 1-MAE: 0.9117
Seed 2024 | Best Epoch:  17 | Best 1-MAE: 0.9118
--------------------------------------------------------------------------------
Mean 1-MAE: 0.9059
Std  1-MAE: 0.0079
Final Result: 0.9059 ± 0.0079

Final saved model: ./save_swintransformer_folder/best_model.pth
